# 🧹 02 - Tiền xử lý dữ liệu

Notebook này demo và chạy pipeline tiền xử lý dữ liệu MC-OCR.

**Nội dung:**
1. Setup môi trường
2. Demo từng bước: parse → clean → normalize → convert
3. Chạy full pipeline
4. Kiểm tra output JSONL
5. Thống kê train/val split

## 1. Setup môi trường

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import sys

REPO_DIR = '/content/document-ai-vn'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/Duc-AnhTp/document-ai-vn.git {REPO_DIR}

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

!pip install -q pandas Pillow scikit-learn

## 2. Demo từng bước tiền xử lý

### 2.1 Parse CSV

In [ ]:
from configs.paths import CSV_PATH, IMAGE_DIR
from src.preprocessing.parse_mcocr import parse_mcocr_csv

samples = parse_mcocr_csv(str(CSV_PATH))
print(f'Tổng số sample: {len(samples)}')

# Xem 1 sample thô
sample_raw = samples[0]
print(f"\nimage_id: {sample_raw['image_id']}")
print(f"texts ({len(sample_raw['texts'])}): {sample_raw['texts'][:3]}")
print(f"labels ({len(sample_raw['labels'])}): {sample_raw['labels'][:3]}")
print(f"polygons: {len(sample_raw['polygons'])} polygon(s)")
print(f"image_quality: {sample_raw['image_quality']}")

### 2.2 Clean & Normalize

In [ ]:
from src.preprocessing.clean_data import clean_sample, clean_text, normalize_label

# Demo clean_text
print('=== Clean text ===')
test_texts = ['  Highlands   Coffee  ', '150.000đ', '   ', '']
for text in test_texts:
    print(f'  "{text}" → "{clean_text(text)}"')

# Demo normalize_label
print('\n=== Normalize label ===')
test_labels = ['SELLER', 'TIMESTAMPS', 'TOTAL_COSTS', 'seller_address', 'OTHER']
for label in test_labels:
    print(f'  "{label}" → "{normalize_label(label)}"')

# Demo clean_sample trên 1 sample
print('\n=== Clean sample ===')
sample_cleaned = clean_sample(sample_raw)
print(f"texts ({len(sample_cleaned['texts'])}): {sample_cleaned['texts'][:3]}")
print(f"labels ({len(sample_cleaned['labels'])}): {sample_cleaned['labels'][:3]}")

### 2.3 Polygon → BBox → Normalize

In [ ]:
from src.utils.bbox import polygon_to_bbox, normalize_bbox
from PIL import Image

# Demo polygon → bbox
if sample_cleaned['polygons']:
    poly_example = sample_cleaned['polygons'][0]
    print(f'Polygon: {poly_example}')
    
    bbox = polygon_to_bbox(poly_example)
    print(f'BBox: {bbox}')
    
    # Normalize cho LayoutLMv3
    img_path = IMAGE_DIR / sample_cleaned['image_id']
    if img_path.exists():
        with Image.open(img_path) as img:
            width, height = img.size
        norm_bbox = normalize_bbox(bbox, width, height)
        print(f'Image size: {width}x{height}')
        print(f'Normalized bbox [0-1000]: {norm_bbox}')

### 2.4 Convert → định dạng model

In [ ]:
from src.preprocessing.run_prepare_data import convert_sample

converted = convert_sample(sample_cleaned)
if converted:
    print(f"image_path: {converted['image_path']}")
    print(f"words ({len(converted['words'])}): {converted['words'][:5]}")
    print(f"boxes ({len(converted['boxes'])}): {converted['boxes'][:3]}")
    print(f"labels ({len(converted['labels'])}): {converted['labels'][:5]}")
else:
    print('Sample bị bỏ qua (ảnh không tồn tại)')

## 3. Chạy full pipeline

In [ ]:
from src.preprocessing.run_prepare_data import main as run_prepare_data

run_prepare_data()

## 4. Kiểm tra output JSONL

In [ ]:
from configs.paths import PROCESSED_DIR
from src.utils.io import load_jsonl

# Kiểm tra file output
for filename in ['all.jsonl', 'train.jsonl', 'val.jsonl']:
    filepath = PROCESSED_DIR / filename
    if filepath.exists():
        records = load_jsonl(filepath)
        print(f'{filename}: {len(records)} samples')
    else:
        print(f'{filename}: KHÔNG TÌM THẤY')

In [ ]:
# Xem mẫu đầu tiên của train.jsonl
train_records = load_jsonl(PROCESSED_DIR / 'train.jsonl')
if train_records:
    record = train_records[0]
    print('Mẫu từ train.jsonl:')
    for key, value in record.items():
        if isinstance(value, list) and len(value) > 5:
            print(f'  {key} ({len(value)} items): {value[:3]} ...')
        else:
            print(f'  {key}: {value}')

## 5. Thống kê train/val split

In [ ]:
from collections import Counter

train_data = load_jsonl(PROCESSED_DIR / 'train.jsonl')
val_data = load_jsonl(PROCESSED_DIR / 'val.jsonl')

print(f'Train: {len(train_data)} samples')
print(f'Val:   {len(val_data)} samples')
print(f'Ratio: {len(val_data)/len(train_data):.2%}')

# Phân bố BIO labels trong train vs val
for split_name, split_data in [('Train', train_data), ('Val', val_data)]:
    all_labels = []
    for record in split_data:
        all_labels.extend(record.get('labels', []))
    counts = Counter(all_labels)
    print(f'\n{split_name} label distribution:')
    for label, count in counts.most_common():
        print(f'  {label}: {count}')